# 🔥 칼로리 소모량 예측 - 최적화 버전 (v3)

---

## 📋 프로젝트 개요

| 항목 | 내용 |
|------|------|
| **목표** | 공동 1등(RMSE 0.10964) → 단독 1등 |
| **전략** | 기존 5개 피처 유지 + 하이퍼파라미터 미세 조정 |
| **기대 성능** | RMSE 0.095~0.102 |

### 개선 사항
1. ✅ 중복 데이터 제거
2. ✅ Ridge Alpha 최적화 (GridSearchCV)
3. ✅ 앙상블 가중치 최적화 (scipy.optimize)
4. ✅ PolynomialFeatures degree 비교 (2 vs 3)

### 고정 사항 (변경 금지)
- 피처: `['Exercise_Duration', 'Gender', 'BPM', 'Age', 'Weight(lb)']`
- 모델: LinearRegression + Ridge 앙상블
- 데이터 경로: Google Drive 지정 경로

---
## 1️⃣ 라이브러리 임포트

In [ ]:
# 기본 라이브러리
import numpy as np
import pandas as pd
import random
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_squared_error

# 최적화
from scipy.optimize import minimize

# 재현성을 위한 시드 고정
def seed_everything(seed=42):
    """모든 랜덤 시드를 고정하여 재현성 보장"""
    random.seed(seed)
    np.random.seed(seed)
    print(f"✅ Random seed 고정: {seed}")

seed_everything(42)

print("\n📦 라이브러리 임포트 완료!")
print(f"   - NumPy: {np.__version__}")
print(f"   - Pandas: {pd.__version__}")

---
## 2️⃣ Google Drive 마운트

In [ ]:
# Google Colab 환경에서 Drive 마운트
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive 마운트 완료!")
    COLAB_ENV = True
except:
    print("⚠️ Google Colab 환경이 아닙니다.")
    print("   로컬 환경에서는 경로를 수정해주세요.")
    COLAB_ENV = False

---
## 3️⃣ 데이터 로드

In [ ]:
# 데이터 경로 설정 (⚠️ 절대 변경 금지)
DATA_PATH = '/content/drive/MyDrive/AI_Projects/calorie_prediction_model/data/'
OUTPUT_PATH = '/content/drive/MyDrive/AI_Projects/calorie_prediction_model/'

# 파일 경로
train_path = DATA_PATH + 'train.csv'
test_path = DATA_PATH + 'test.csv'
submit_path = DATA_PATH + 'sample_submission.csv'
output_file = OUTPUT_PATH + 'submission_v3_optimized.csv'

print("📂 데이터 경로 설정:")
print(f"   Train: {train_path}")
print(f"   Test: {test_path}")
print(f"   Submit: {submit_path}")
print(f"   Output: {output_file}")

In [ ]:
# 데이터 로드
try:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    submit = pd.read_csv(submit_path)
    
    print("✅ 데이터 로드 완료!")
    print(f"\n📊 데이터 크기:")
    print(f"   Train: {train.shape[0]:,}개 행, {train.shape[1]}개 컬럼")
    print(f"   Test: {test.shape[0]:,}개 행, {test.shape[1]}개 컬럼")
    print(f"   Submit: {submit.shape[0]:,}개 행, {submit.shape[1]}개 컬럼")
    
except FileNotFoundError as e:
    print(f"❌ 파일을 찾을 수 없습니다!")
    print(f"   에러: {e}")
    print("\n💡 해결 방법:")
    print("   1. Google Drive가 마운트되었는지 확인")
    print("   2. 파일 경로가 정확한지 확인")
    raise

---
## 4️⃣ EDA (탐색적 데이터 분석)

간단한 데이터 탐색을 통해 데이터 품질을 확인합니다.

In [ ]:
# 데이터 미리보기
print("📋 Train 데이터 상위 5개 행:")
display(train.head())

print("\n📋 컬럼 목록:")
print(f"   Train: {list(train.columns)}")
print(f"   Test: {list(test.columns)}")

In [ ]:
# 데이터 품질 확인
print("🔍 데이터 품질 확인:")
print(f"\n1. 결측치:")
print(f"   Train: {train.isnull().sum().sum()}개")
print(f"   Test: {test.isnull().sum().sum()}개")

print(f"\n2. 중복 데이터:")
duplicates = train.duplicated().sum()
print(f"   Train 중복 행: {duplicates}개 ({duplicates/len(train)*100:.2f}%)")

print(f"\n3. 목표 변수 (Calories_Burned) 통계:")
print(train['Calories_Burned'].describe())

---
## 5️⃣ 중복 데이터 제거

**왜 중복 제거가 중요한가?**
- 중복 데이터는 모델이 특정 패턴에 과적합되게 만듦
- 교차검증 시 데이터 누출(data leakage) 위험
- 중복 제거로 더 일반화된 모델 학습 가능

**예상 효과:** RMSE 약 2~5% 개선

In [ ]:
# 중복 제거 전 데이터 크기 저장
original_size = len(train)

# 중복 제거 실행
train = train.drop_duplicates()

# 결과 확인
new_size = len(train)
removed = original_size - new_size

print("✅ 중복 데이터 제거 완료!")
print(f"\n📊 결과:")
print(f"   제거 전: {original_size:,}개")
print(f"   제거 후: {new_size:,}개")
print(f"   제거된 행: {removed:,}개 ({removed/original_size*100:.2f}%)")

# 인덱스 리셋
train = train.reset_index(drop=True)
print(f"\n✅ 인덱스 리셋 완료")

---
## 6️⃣ 피처 선택 및 전처리

### 사용 피처 (5개, 변경 금지)
1. `Exercise_Duration` - 운동 시간 (분)
2. `Gender` - 성별 (F/M)
3. `BPM` - 심박수
4. `Age` - 나이
5. `Weight(lb)` - 체중 (파운드)

**주의:** BMI, Body_Temperature, Weight_Status는 사용하지 않음 (검증된 최적 조합 유지)

In [ ]:
# 피처 선택 (⚠️ 절대 변경 금지)
features = ['Exercise_Duration', 'Gender', 'BPM', 'Age', 'Weight(lb)']

print("📌 선택된 피처 (5개):")
for i, feat in enumerate(features, 1):
    print(f"   {i}. {feat}")

# 피처 및 타겟 분리
X = train[features].copy()
y = train['Calories_Burned'].copy()
X_test = test[features].copy()

print(f"\n📊 데이터 shape:")
print(f"   X: {X.shape}")
print(f"   y: {y.shape}")
print(f"   X_test: {X_test.shape}")

In [ ]:
# Gender 인코딩 (LabelEncoder)
le = LabelEncoder()
X['Gender'] = le.fit_transform(X['Gender'])
X_test['Gender'] = le.transform(X_test['Gender'])

print("✅ Gender 인코딩 완료!")
print(f"   매핑: {dict(zip(le.classes_, range(len(le.classes_))))}")

# 인코딩 후 데이터 확인
print(f"\n📋 인코딩 후 데이터 타입:")
print(X.dtypes)

---
## 7️⃣ PolynomialFeatures Degree 비교

**목적:** degree=2와 degree=3 중 최적 값 선택

**비교 방법:** 5-Fold 교차검증으로 RMSE 비교

| Degree | 피처 수 (5개 기준) | 특징 |
|--------|-------------------|------|
| 2 | ~15개 | 2차 상호작용만 |
| 3 | ~25개 | 3차 상호작용까지 |

In [ ]:
print("🔬 PolynomialFeatures Degree 비교 (2 vs 3)")
print("=" * 50)

degree_results = {}

for degree in [2, 3]:
    # PolynomialFeatures 변환
    poly_test = PolynomialFeatures(
        degree=degree, 
        interaction_only=True, 
        include_bias=False
    )
    X_poly_test = poly_test.fit_transform(X)
    
    # Ridge 모델로 교차검증 (alpha=1.0, 기본값)
    ridge_test = Ridge(alpha=1.0)
    cv_scores = cross_val_score(
        ridge_test,
        X_poly_test,
        y,
        cv=5,
        scoring='neg_root_mean_squared_error'
    )
    
    mean_rmse = -cv_scores.mean()
    std_rmse = cv_scores.std()
    
    degree_results[degree] = {
        'n_features': X_poly_test.shape[1],
        'cv_rmse': mean_rmse,
        'cv_std': std_rmse
    }
    
    print(f"\nDegree = {degree}:")
    print(f"   피처 수: {X_poly_test.shape[1]}개")
    print(f"   CV RMSE: {mean_rmse:.5f} (+/- {std_rmse:.5f})")

# 최적 degree 선택
best_degree = min(degree_results.keys(), key=lambda d: degree_results[d]['cv_rmse'])
print(f"\n" + "=" * 50)
print(f"🏆 최적 Degree: {best_degree}")
print(f"   CV RMSE: {degree_results[best_degree]['cv_rmse']:.5f}")

---
## 8️⃣ Ridge Alpha 최적화 (GridSearchCV)

**목적:** Ridge 정규화 강도(alpha) 최적화

**탐색 범위:** [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]

**방법:** 5-Fold 교차검증

In [ ]:
# 최적 degree로 PolynomialFeatures 적용
print(f"📌 선택된 Degree: {best_degree}")

poly = PolynomialFeatures(
    degree=best_degree, 
    interaction_only=True, 
    include_bias=False
)

X_poly = poly.fit_transform(X)
X_test_poly = poly.transform(X_test)

print(f"\n✅ PolynomialFeatures 변환 완료!")
print(f"   원본 피처: {X.shape[1]}개")
print(f"   변환 후: {X_poly.shape[1]}개")

In [ ]:
print("🔬 Ridge Alpha 최적화 (GridSearchCV)")
print("=" * 50)

# Alpha 탐색 범위
param_grid = {
    'alpha': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]
}

print(f"탐색 범위: {param_grid['alpha']}")

# GridSearchCV 실행
ridge_cv = GridSearchCV(
    Ridge(),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    return_train_score=True
)

ridge_cv.fit(X_poly, y)

# 결과 출력
print(f"\n✅ GridSearchCV 완료!")
print(f"\n🏆 최적 Alpha: {ridge_cv.best_params_['alpha']}")
print(f"   최적 CV RMSE: {-ridge_cv.best_score_:.5f}")

# 모든 결과 출력
print(f"\n📊 Alpha별 결과:")
print("-" * 40)
results_df = pd.DataFrame(ridge_cv.cv_results_)
for _, row in results_df.iterrows():
    alpha = row['param_alpha']
    mean_score = -row['mean_test_score']
    std_score = row['std_test_score']
    marker = " 🏆" if alpha == ridge_cv.best_params_['alpha'] else ""
    print(f"   Alpha={alpha:<6}: RMSE={mean_score:.5f} (+/- {std_score:.5f}){marker}")

# 최적 모델 저장
best_alpha = ridge_cv.best_params_['alpha']
ridge_optimal = ridge_cv.best_estimator_

---
## 9️⃣ 모델 학습

### 사용 모델
1. **LinearRegression** - 기본 선형 회귀
2. **Ridge (최적 alpha)** - L2 정규화 선형 회귀

In [ ]:
print("🚀 모델 학습 시작")
print("=" * 50)

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_poly, y)
lr_pred_train = lr.predict(X_poly)
lr_rmse = np.sqrt(mean_squared_error(y, lr_pred_train))

print(f"\n1️⃣ LinearRegression")
print(f"   Train RMSE: {lr_rmse:.5f}")

# 2. Ridge (최적 alpha)
ridge = Ridge(alpha=best_alpha)
ridge.fit(X_poly, y)
ridge_pred_train = ridge.predict(X_poly)
ridge_rmse = np.sqrt(mean_squared_error(y, ridge_pred_train))

print(f"\n2️⃣ Ridge (alpha={best_alpha})")
print(f"   Train RMSE: {ridge_rmse:.5f}")

print(f"\n✅ 모델 학습 완료!")

---
## 🔟 앙상블 가중치 최적화

**목적:** LR과 Ridge의 최적 앙상블 가중치 찾기

**방법:** scipy.optimize.minimize + 5-Fold 교차검증

**제약 조건:** w1 + w2 = 1 (가중치 합 = 1)

In [ ]:
print("🔬 앙상블 가중치 최적화")
print("=" * 50)

def ensemble_rmse(weights, lr_pred, ridge_pred, y_true):
    """앙상블 예측의 RMSE 계산"""
    # 반올림 적용
    ensemble_pred = weights[0] * np.round(lr_pred) + weights[1] * np.round(ridge_pred)
    return np.sqrt(mean_squared_error(y_true, ensemble_pred))

# 교차검증으로 최적 가중치 찾기
kf = KFold(n_splits=5, shuffle=True, random_state=42)

all_weights = []
all_scores = []

print("\n5-Fold 교차검증 진행 중...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_poly), 1):
    # 데이터 분할
    X_train_fold = X_poly[train_idx]
    X_val_fold = X_poly[val_idx]
    y_train_fold = y.iloc[train_idx].values
    y_val_fold = y.iloc[val_idx].values
    
    # 모델 학습
    lr_fold = LinearRegression().fit(X_train_fold, y_train_fold)
    ridge_fold = Ridge(alpha=best_alpha).fit(X_train_fold, y_train_fold)
    
    # 검증 예측
    lr_val_pred = lr_fold.predict(X_val_fold)
    ridge_val_pred = ridge_fold.predict(X_val_fold)
    
    # 최적 가중치 탐색
    result = minimize(
        ensemble_rmse,
        x0=[0.5, 0.5],  # 초기값
        args=(lr_val_pred, ridge_val_pred, y_val_fold),
        method='SLSQP',
        bounds=[(0, 1), (0, 1)],
        constraints={'type': 'eq', 'fun': lambda w: w[0] + w[1] - 1}
    )
    
    all_weights.append(result.x)
    all_scores.append(result.fun)
    
    print(f"   Fold {fold}: LR={result.x[0]:.3f}, Ridge={result.x[1]:.3f}, RMSE={result.fun:.5f}")

# 평균 가중치 계산
avg_weights = np.mean(all_weights, axis=0)
avg_score = np.mean(all_scores)

# 가중치 정규화 (합이 1이 되도록)
best_lr_weight = avg_weights[0] / sum(avg_weights)
best_ridge_weight = avg_weights[1] / sum(avg_weights)

print(f"\n" + "=" * 50)
print(f"🏆 최적 앙상블 가중치:")
print(f"   LR: {best_lr_weight:.4f}")
print(f"   Ridge: {best_ridge_weight:.4f}")
print(f"   평균 CV RMSE: {avg_score:.5f}")

# 기존 가중치와 비교
print(f"\n📊 기존 vs 최적화:")
print(f"   기존: LR=0.600, Ridge=0.400")
print(f"   최적: LR={best_lr_weight:.3f}, Ridge={best_ridge_weight:.3f}")

---
## 1️⃣1️⃣ Train 성능 검증

In [ ]:
print("📊 Train 성능 검증")
print("=" * 50)

# 각 모델 예측 (반올림 적용)
lr_pred_rounded = np.round(lr_pred_train)
ridge_pred_rounded = np.round(ridge_pred_train)

# 기존 가중치 앙상블 (0.6, 0.4)
old_ensemble = 0.6 * lr_pred_rounded + 0.4 * ridge_pred_rounded
old_rmse = np.sqrt(mean_squared_error(y, old_ensemble))

# 최적 가중치 앙상블
new_ensemble = best_lr_weight * lr_pred_rounded + best_ridge_weight * ridge_pred_rounded
new_rmse = np.sqrt(mean_squared_error(y, new_ensemble))

print(f"\n1️⃣ 개별 모델 (Train RMSE):")
print(f"   LinearRegression: {lr_rmse:.5f}")
print(f"   Ridge (alpha={best_alpha}): {ridge_rmse:.5f}")

print(f"\n2️⃣ 앙상블 (Train RMSE):")
print(f"   기존 (0.6/0.4): {old_rmse:.5f}")
print(f"   최적화 ({best_lr_weight:.3f}/{best_ridge_weight:.3f}): {new_rmse:.5f}")

# 개선율 계산
baseline_rmse = 0.10964
improvement = (baseline_rmse - new_rmse) / baseline_rmse * 100

print(f"\n3️⃣ 기존 모델 대비:")
print(f"   기존 최고 성능: {baseline_rmse:.5f}")
print(f"   현재 성능: {new_rmse:.5f}")
print(f"   개선율: {improvement:.2f}%")

In [ ]:
# 교차검증 RMSE 계산
print("📊 교차검증 성능 (5-Fold)")
print("=" * 50)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_poly), 1):
    # 데이터 분할
    X_train_fold = X_poly[train_idx]
    X_val_fold = X_poly[val_idx]
    y_train_fold = y.iloc[train_idx].values
    y_val_fold = y.iloc[val_idx].values
    
    # 모델 학습
    lr_fold = LinearRegression().fit(X_train_fold, y_train_fold)
    ridge_fold = Ridge(alpha=best_alpha).fit(X_train_fold, y_train_fold)
    
    # 검증 예측 (반올림 적용)
    lr_val_pred = np.round(lr_fold.predict(X_val_fold))
    ridge_val_pred = np.round(ridge_fold.predict(X_val_fold))
    
    # 앙상블 예측
    ensemble_pred = best_lr_weight * lr_val_pred + best_ridge_weight * ridge_val_pred
    
    # RMSE 계산
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, ensemble_pred))
    cv_rmse_scores.append(fold_rmse)
    
    print(f"   Fold {fold}: RMSE = {fold_rmse:.5f}")

cv_mean = np.mean(cv_rmse_scores)
cv_std = np.std(cv_rmse_scores)

print(f"\n" + "=" * 50)
print(f"📈 CV RMSE: {cv_mean:.5f} (+/- {cv_std:.5f})")

---
## 1️⃣2️⃣ Test 예측

In [ ]:
print("🎯 Test 데이터 예측")
print("=" * 50)

# 각 모델로 Test 예측
lr_pred_test = lr.predict(X_test_poly)
ridge_pred_test = ridge.predict(X_test_poly)

# 반올림 적용
lr_pred_test_rounded = np.round(lr_pred_test)
ridge_pred_test_rounded = np.round(ridge_pred_test)

# 최적 가중치로 앙상블
final_pred = best_lr_weight * lr_pred_test_rounded + best_ridge_weight * ridge_pred_test_rounded

print("✅ Test 예측 완료!")
print(f"\n📊 예측값 통계:")
print(f"   Min:  {final_pred.min():.1f}")
print(f"   Max:  {final_pred.max():.1f}")
print(f"   Mean: {final_pred.mean():.1f}")
print(f"   Std:  {final_pred.std():.1f}")

# 음수 값 체크
neg_count = (final_pred < 0).sum()
if neg_count > 0:
    print(f"\n⚠️ 음수 예측값: {neg_count}개")
    final_pred = np.maximum(final_pred, 0)  # 음수 → 0으로 변환
    print("   → 0으로 변환 완료")

---
## 1️⃣3️⃣ Submission 저장

In [ ]:
print("💾 Submission 파일 저장")
print("=" * 50)

# submission 데이터프레임에 예측값 저장
submit['Calories_Burned'] = final_pred

# 파일 저장
try:
    submit.to_csv(output_file, index=False)
    print(f"\n✅ 저장 완료!")
    print(f"   파일명: submission_v3_optimized.csv")
    print(f"   경로: {output_file}")
except Exception as e:
    print(f"❌ 저장 실패: {e}")
    # 대체 경로
    alt_path = 'submission_v3_optimized.csv'
    submit.to_csv(alt_path, index=False)
    print(f"   대체 경로에 저장: {alt_path}")

# 미리보기
print(f"\n📋 Submission 미리보기 (상위 5개):")
display(submit.head())

print(f"\n📊 Submission 통계:")
print(submit['Calories_Burned'].describe())

---
## 1️⃣4️⃣ 최종 결과 요약

In [ ]:
print("="*60)
print("=== 칼로리 소모량 예측 - 최적화 버전 ===")
print("="*60)

print(f"""
📊 데이터:
   - Train (중복 제거 전): {original_size:,}개
   - Train (중복 제거 후): {new_size:,}개
   - Test: {len(test):,}개

🔧 최적 하이퍼파라미터:
   - Ridge Alpha: {best_alpha}
   - PolynomialFeatures Degree: {best_degree}
   - 앙상블 가중치: LR={best_lr_weight:.4f}, Ridge={best_ridge_weight:.4f}
   - 피처 개수: {X.shape[1]}개 → {X_poly.shape[1]}개 (Polynomial 후)

📈 성능 (RMSE):
   - Train: {new_rmse:.5f}
   - CV (5-Fold): {cv_mean:.5f} (+/- {cv_std:.5f})
   - 기존 대비: {improvement:.2f}% {'개선' if improvement > 0 else '저하'}

💾 저장:
   - 파일: submission_v3_optimized.csv
   - 경로: {output_file}

🎯 예측 통계:
   - Min: {final_pred.min():.1f}
   - Max: {final_pred.max():.1f}
   - Mean: {final_pred.mean():.1f}
   - Std: {final_pred.std():.1f}
""")

print("="*60)
print("✅ 모든 작업이 완료되었습니다!")
print("="*60)

---
## 📝 변경 이력

| 버전 | 날짜 | 변경 사항 | RMSE |
|------|------|----------|------|
| v1 | - | 기본 5피처 + LR/Ridge | 0.10964 |
| v2 | - | +BMI, Body_Temp, Weight_Status | (테스트 중) |
| **v3** | - | 하이퍼파라미터 최적화 | **(목표: <0.10)** |

---

## 🔗 참고

- [Scikit-learn Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)
- [Scikit-learn GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [SciPy Optimize](https://docs.scipy.org/doc/scipy/reference/optimize.html)